## Fifth step

In `fifth_step.ipynb`, we will use the records filtered (or not) in `fourth_step.ipynb` and filter the taxonomists. Taxonomists are the individuals responsible for the taxonomic identification of the plant records in our collection; it is crucial that this identification be accurate, given that our labels will be used to train an artificial intelligence, meaning that if the labels are incorrect, the AI ​​will learn those errors as well.

Therefore, we will select the taxonomists with the highest level of confidence based on the number of identifications they have made, considering that the more identifications a taxonomist has made within a given family, the lower the likelihood of an incorrect identification.

Following the same convention as the previous step, the name of the new field will be `identifiedby_upd`, since the original field is `identifiedby`.

If you do not wish to perform this filtering, continue using the `identifiedby` field instead of `identifiedby_upd` for subsequent filtering operations.

In [ ]:
from library import *

specieslink, db_config = configure()

In [ ]:
def taxonomists(table, db_config):
    conn = mysql_conn.connect(**db_config)
    cursor = conn.cursor()

    requirement = "country_upd"
    requirement2 = "stateprovince_upd"
    requirement3 = "barcode_upd"
    column = "identifiedby"

    sql = f"""SELECT {column} FROM {table} WHERE {column} IS NOT NULL AND {requirement} IS NOT NULL AND {requirement2} IS NOT NULL AND {requirement3} IS NOT NULL GROUP BY {column}"""

    cursor.execute(sql)
    authors = [row[0] for row in cursor.fetchall()]

    cursor.close()
    conn.close()  

    return authors

In [ ]:
ranking = int(input("how many of the most frequent taxonomists do you want to display?").strip())
similar = int(input("fuzzy similarity (0-100%): ").strip())
table = "biodiversity_records"
txt = input(".txt path to save the clustering results: ").strip()

if not ranking:
    ranking = 5
if similar < 0 or similar > 100:
    similar = 60
if not txt:
    txt = "deduplicacao.txt"

authors = taxonomists(table, db_config)

df = pd.DataFrame({"identifiedby": authors})

csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

processing(csv_buffer, ranking, similar, txt)

In [ ]:
def build_taxonomists_map(txt_filename, top_n):
    taxonomists_map = {}
    
    if not os.path.exists(txt_filename):
        print(f"file '{txt_filename}' not found")
        return taxonomists_map
    
    all_clusters = []
    current_main = None # main name
    current_count = 0 # count of occurences
    current_variations = set() # name variation of main name
    
    with open(txt_filename, 'r', encoding='utf-8') as file:
        for line in file:
            line_stripped = line.strip()
            if not line_stripped: # if empty
                continue
            
            # first name
            if " : " in line and "occurence(s)" in line:
                if current_main:
                    all_clusters.append((current_count, current_main, list(current_variations))) # save current taxonomist before parsing next
                
                parts = line.split(" : ")
                current_main = parts[0].strip()
                
                count_str = parts[1].replace("occurence(s)", "").strip() # gets occurences
                current_count = int(count_str) if count_str.isdigit() else 0 # saves as int
                
                current_variations = set() # restart variations for next main name
            
            # variations
            elif line_stripped.startswith("-"):
                var_name = line_stripped[1:].strip()
                var_name = re.sub(r'\s*\(\d+\)$', '', var_name).strip()
                
                if current_main is not None:
                    current_variations.add(var_name) # add to current main name cluster
                    
    # last taxonomist
    if current_main:
        all_clusters.append((current_count, current_main, list(current_variations)))

    # top x
    all_clusters.sort(key=lambda x: x[0], reverse=True) # order by occurences
    top_clusters = all_clusters[:top_n]
    
    for count, main_name, variations in top_clusters:
        taxonomists_map[main_name] = variations
        
    return taxonomists_map

In [ ]:
def pipeline_filtering(field_input,table,filter_field,filter_values,update_value,db_config):
    if not field_input:
        print("field_input not informed")
        return

    conn = mysql_conn.connect(**db_config)
    cursor = conn.cursor()

    try:
        sql = f"ALTER TABLE {table} ADD COLUMN {field_input} TEXT"
        cursor.execute(sql)
        conn.commit()
        print(f"field '{field_input}' created with succes in table '{table}'")
    except Exception:
        pass
    finally:
        cursor.close()
        conn.close()

    for value in filter_values:
        specieslink.update_records(
            filters={filter_field: value},
            update_values={field_input: update_value},
            db_config=db_config,
            table=table
        )


In [ ]:
import re
TAXONOMISTS_MAP = build_taxonomists_map(txt_filename=txt, top_n=ranking)

field_input = input("specify the name you want the new field to have (do not create the field manually when running via the pipeline!)").strip()
table = "biodiversity_records"

for default_state, variants in TAXONOMISTS_MAP.items():
    for variante in variants:
        update_input = f"{field_input}={default_state}"
        filters_input = f"identifiedby={variante}"

        print(f"executing specieslink.update_records(filters={filters_input}, update_values={update_input}, table={table})...\n")


    pipeline_filtering(
        field_input=field_input,
        table=table,
        filter_field="identifiedby",
        filter_values=variants,
        update_value=default_state,
        db_config=db_config
    )


In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

sql = """
SELECT
    COUNT(*) AS total_records,
    SUM(country_upd IS NOT NULL) AS count_country,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL) AS count_country_n_state,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL AND barcode_upd IS NOT NULL) as count_country_state_n_barcode,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL AND barcode_upd IS NOT NULL AND identifiedby_upd IS NOT NULL) as count_country_state_barcode_n_taxonomist
FROM biodiversity_records
"""

cursor.execute(sql)
total_records, records_country, records_country_state, records_country_state_barcode, records_country_state_barcode_taxonomist = cursor.fetchone()

cursor.close()
conn.close()

In [ ]:
plt.figure(figsize=(6, 4))

x = [0]

plt.bar(
    x,
    [total_records],
    alpha=0.5,
    width=0.15,
    label='total'
)

plt.bar(
    x,
    [records_country],
    alpha=0.5,
    width=0.15,
    label='country_upd'
)

plt.bar(
    x,
    [records_country_state],
    width=0.15,
    label='country_upd e stateprovince_upd'
)

plt.bar(
    x,
    [records_country_state_barcode],
    width=0.15,
    label='country_upd, stateprovince_upd e barcode_upd'
)

plt.bar(
    x,
    [records_country_state_barcode_taxonomist],
    width=0.15,
    label='country_upd, stateprovince_upd, barcode_upd e identifiedby_upd'
)

plt.xlim(-0.2, 0.3)
plt.ylabel('total records')
plt.title('sample of records to be used')
plt.legend()

plt.text(0, total_records, str(total_records), ha='center', va='bottom')
plt.text(0, records_country, str(records_country), ha='center', va='bottom')
plt.text(0, records_country_state, str(records_country_state), ha='center', va='bottom')
plt.text(0, records_country_state_barcode, str(records_country_state_barcode), ha='center', va='bottom')
plt.text(0, records_country_state_barcode, str(records_country_state_barcode_taxonomist), ha='center', va='bottom')

plt.tight_layout()
plt.show()